# Codificacion de valores faltantes 

In [ ]:
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import numpy as np

from src.pandas_accessors import MissingMethods
from src.data_loader import load_all_datasets

datasets = load_all_datasets()

pima_dataset =datasets['pima']

<div style="background-color: rgba(255, 235, 59, 0.12); padding: 20px;">

### Advertencia

Al igual que cada persona es una nueva puerta a un mundo diferente, los **valores faltantes** existen en diferentes formas y colores. Al trabajar con valores faltantes será crítico entender sus distintas representaciones. A pesar de que el conjunto de datos de trabajo pareciera que no contiene valores faltantes, deberás ser capaz de ir más allá de lo observado a simple vista para remover el manto tras el cual se esconde lo desconocido.

</div>

### Valores Comunmente asociados a valores faltantes

#### Cadenas

In [ ]:
common_na_strings = (
    "missing",
    "NA",
    "N A",
    "N/A",
    "#N/A",
    "NA ",
    " NA",
    "N/ A",
    "N /A",
    "N / A",
    "na",
    "n/a",
    "n /a",
    "n / a",
    "n/ a",
    "a / a",
    "Null",
    "null",
    "",
    "?",
    "*",
    "."
    ","
)

#### Numeros

In [ ]:
common_na_numbers = (-9,-99, -999, -9999, 9999, 66, 77, 88, -1)

### Como encontrar los valores comunmente asociados a valores faltantes?

In [ ]:
missing_data_example_df = pd.DataFrame.from_dict(
    dict(
        x = [1, 2, "NA", -99, -98, -99],
        y = ["A", "N/A", "NA", "E", "F", "G"],
        z = [-100, -99, -98, -101, 1, -1]
    )
)      

missing_data_example_df

,x,y,z
0,1,A,-100
1,2,N/A,-99
2,NA,NA,-98
3,-99,E,-101
4,-98,F,1
5,-99,G,-1


In [ ]:
missing_data_example_df.missing.number_missing()

np.int64(0)

#### Revisar Tipos de datos

In [ ]:
missing_data_example_df.dtypes

# Obtener un objeto en alguna columna puede ser una pista de que valores faltantes están codificados como cadenas de texto. En este caso, la columna `x` tiene un tipo de dato `object`, lo que indica que contiene valores de tipo cadena. Esto puede ser una señal de que los valores faltantes están representados por cadenas como "NA" o "N/A".

x    object
y       str
z     int64
dtype: object

#### Revisar valores unicos de datos

In [ ]:
print(f"Valores únicos en la columna x: {missing_data_example_df.x.unique()}")
print(f"Valores únicos en la columna y: {missing_data_example_df.y.unique()}")
print(f"Valores únicos en la columna z: {missing_data_example_df.z.unique()}")

Valores únicos en la columna x: [1 2 'NA' -99 -98]
Valores únicos en la columna y: <StringArray>
['A', 'N/A', 'NA', 'E', 'F', 'G']
Length: 6, dtype: str
Valores únicos en la columna z: [-100  -99  -98 -101   -1]


In [ ]:
(
    missing_data_example_df
    .select_dtypes(object)
    .apply(pd.unique)
)    

/tmp/ipykernel_9917/3187778216.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(object)


x     [1, 2, NA, -99, -98]
y    [A, N/A, NA, E, F, G]
dtype: object

#### Sustituyendo valores comunmente asociados a valores faltantes

##### Sustitucion desde la lectura de datos

In [ ]:
missing_data_example_df_clean = missing_data_example_df.replace(
    [-99, -1, "NA", "N/A", "missing", "null", "Null"],
    np.nan
)

missing_data_example_df_clean

,x,y,z
0,1,A,-100.0
1,2,NaN,NaN
2,NaN,NaN,-98.0
3,NaN,E,-101.0
4,-98,F,NaN
5,NaN,G,NaN


##### Sustitucion Global

In [ ]:
missing_data_example_df_clean = missing_data_example_df.replace(
    to_replace=["NA", "N/A", "missing", "null", "Null"],
    value=np.nan
)

numeric_values = missing_data_example_df_clean.apply(
    pd.to_numeric,
    errors="coerce"
)

missing_data_example_df_clean = missing_data_example_df_clean.mask(
    numeric_values < 0,
    np.nan
)


missing_data_example_df_clean

,x,y,z
0,1,A,NaN
1,2,NaN,NaN
2,NaN,NaN,NaN
3,NaN,E,NaN
4,NaN,F,1.0
5,NaN,G,NaN


##### Sustitucion Dirigida

In [ ]:
missing_data_example_df_clean = missing_data_example_df.replace(
    to_replace={
        "x" : {
            -99: np.nan,
            -98: np.nan
        },
        "z" : {
            -99: np.nan,
        }
    }
)

missing_data_example_df_clean

,x,y,z
0,1,A,-100.0
1,2,N/A,NaN
2,NA,NA,-98.0
3,NaN,E,-101.0
4,NaN,F,1.0
5,NaN,G,-1.0
